# Geographic Prediction with Random Forests

## Load libraries

In [1]:
import numpy as np
import allel
from tqdm import tqdm
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import json
from typing import List

## Loading the data

In [2]:
metadata = pd.read_csv("metadata_cleaned.csv")
metadata

,sample_id,sample,sra_run,country,site,year,Study,collaborator,Country_x,Region_x,Country_y,Region_y,Country,Region
0,ERR1081237,FP0008-C,ERR1081237,Mauritania,NaN,2014.0,1147-PF-MR-CONWAY,NaN,Mauritania,Western Africa,Mauritania,Western Africa,Mauritania,Western Africa
1,ERR1081238,FP0009-C,ERR1081238,Mauritania,NaN,2014.0,1147-PF-MR-CONWAY,NaN,Mauritania,Western Africa,Mauritania,Western Africa,Mauritania,Western Africa
2,ERR2889621,FP0010-CW,ERR2889621,Mauritania,NaN,2014.0,1147-PF-MR-CONWAY,NaN,Mauritania,Western Africa,Mauritania,Western Africa,Mauritania,Western Africa
3,ERR2889624,FP0011-CW,ERR2889624,Mauritania,NaN,2014.0,1147-PF-MR-CONWAY,NaN,Mauritania,Western Africa,Mauritania,Western Africa,Mauritania,Western Africa
4,ERR2889627,FP0012-CW,ERR2889627,Mauritania,NaN,2014.0,1147-PF-MR-CONWAY,NaN,Mauritania,Western Africa,Mauritania,Western Africa,Mauritania,Western Africa
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19838,ERR3486394,SPT43377,ERR3486394,Kenya,NaN,2017.0,1132-PF-K1000G-DBS-KE-BEJON,NaN,Kenya,Eastern Africa,Kenya,Eastern Africa,Kenya,Eastern Africa
19839,ERR3486397,SPT43381,ERR3486397,Kenya,NaN,2017.0,1132-PF-K1000G-DBS-KE-BEJON,NaN,Kenya,Eastern Africa,Kenya,Eastern Africa,Kenya,Eastern Africa
19840,ERR3486399,SPT43390,ERR3486399,Kenya,NaN,2018.0,1132-PF-K1000G-DBS-KE-BEJON,NaN,Kenya,Eastern Africa,Kenya,Eastern Africa,Kenya,Eastern Africa
19841,ERR3486381,SPT43391,ERR3486381,Kenya,NaN,2018.0,1132-PF-K1000G-DBS-KE-BEJON,NaN,Kenya,Eastern Africa,Kenya,Eastern Africa,Kenya,Eastern Africa


## Visualizing Data Distributions

In [3]:
px.bar(
    metadata['country'].value_counts(),
    template='simple_white',
    title='Number of samples per country'
)

In [4]:
px.bar(
    metadata['Region'].value_counts(),
    template='simple_white',
    title='Number of samples per region'
)

## Split data into training and test sets

In [5]:
# split data into 80:20 train test
from sklearn.model_selection import train_test_split
train_metadata, test_metadata = train_test_split(metadata, test_size=0.2, random_state=42, stratify=metadata['Region'])

In [6]:
input_file = 'variants.vcf.gz'
callset = allel.read_vcf(input_file)

In [7]:
def subset_callset_by_sample(callset: dict,samples: List[str]) -> dict:
    sample_index = [i for i,s in enumerate(callset['samples']) if s in samples]
    return {
        "samples":callset['samples'][sample_index],
        "calldata/GT": callset['calldata/GT'][:,sample_index,:],
        "variants/ALT": callset['variants/ALT'],
        "variants/CHROM": callset['variants/CHROM'],
        "variants/FILTER_PASS": callset['variants/FILTER_PASS'],
        "variants/ID": callset['variants/ID'],
        "variants/POS": callset['variants/POS'],
        "variants/QUAL": callset['variants/QUAL'],
        "variants/REF": callset['variants/REF'],
        "variants/GENE": callset.get('variants/GENE',None),
        "variants/AA": callset.get('variants/AA',None),
    }

train_callset = subset_callset_by_sample(callset,train_metadata['sample'].tolist())
test_callset = subset_callset_by_sample(callset,test_metadata['sample'].tolist())

In [8]:
rows = []
for i in tqdm(range(len(train_callset['variants/POS']))):
    row = [x[1] if x[1]!=-1 else np.nan for x in train_callset['calldata/GT'][i]]
    rows.append(row)

X_train = np.array(rows).T

100%|██████████| 56/56 [00:00<00:00, 312.45it/s]


In [9]:
sample2region = dict(zip(metadata['sample'],metadata['Region']))
y_train = [sample2region[s] for s in train_callset['samples']]



## Set up Random Forest model

In [10]:
from sklearn.ensemble import HistGradientBoostingClassifier
clf = HistGradientBoostingClassifier(max_iter=100).fit(X_train, y_train)
clf.score(X_train, y_train)

0.9331611440090715

In [11]:
rows = []
for i in tqdm(range(len(test_callset['variants/POS']))):
    row = [x[1] if x[1]!=-1 else np.nan for x in test_callset['calldata/GT'][i]]
    rows.append(row)

X_test = np.array(rows).T

y_test = [sample2region[s] for s in test_callset['samples']]

clf.score(X_test, y_test)

100%|██████████| 56/56 [00:00<00:00, 1286.92it/s]


0.9062736205593348

## Look at prediction by country

In [75]:
sample2country = dict(zip(metadata['sample'],metadata['country']))

In [124]:
y_country_train = np.array([sample2country[s] for s in train_callset['samples']])
clf_country = HistGradientBoostingClassifier(max_iter=100).fit(X_train, y_country_train)

In [127]:
y_country_test = np.array([sample2country[s] for s in test_callset['samples']])
clf_country.score(X_test, y_country_test)

0.4386495338876291

## Probability visualisation

In [ ]:
geojson = json.load(open("world.geojson"))

In [152]:
df = pd.DataFrame({'country':clf_country.classes_,'probability':clf_country.predict_proba(X_test)[203]})
subgeojson = {'type': 'FeatureCollection', 'features': [x for x in geojson['features'] if x['properties']['ADMIN'] in df['country'].values]}

In [153]:
fig = go.Figure(data=go.Choropleth(
    locations=df['country'],locationmode="country names",
    z=df['probability'], colorscale='Reds',
    marker_line_color='darkgray',
    marker_line_width=0.5,
    )
)
fig.update_layout(
    title_text='Geographic source probability',
    geo=dict(
        showframe=False,
        showcoastlines=False,
        projection_type='equirectangular'
    ),
)
fig.update_layout(
    margin=dict(l=20, r=20, t=60, b=20),
)
fig.show()

/tmp/ipykernel_3672991/2156714765.py:1: DeprecationWarning:

The library used by the *country names* `locationmode` option is changing in an upcoming version. Country names in existing plots may not work in the new version. To ensure consistent behavior, consider setting `locationmode` to *ISO-3*.



In [147]:
np.where(pd.Series(y_country_test)=='Thailand')

(array([ 201,  202,  203,  204,  205,  206,  207,  208,  209,  210,  211,
         212,  213,  214,  215,  216,  217,  218,  219,  220,  221,  222,
         223,  224,  225,  226,  227,  228,  229,  230,  231,  232,  233,
         234,  235,  236,  237,  238,  239,  240,  241,  242,  243,  244,
         245,  246,  247,  248,  249,  250,  251,  252,  253,  254,  255,
         256,  257,  258,  259,  260,  261,  262,  263,  264,  265,  266,
         267,  268,  269,  270,  271,  272,  273,  274,  275,  276,  277,
         278,  279,  280,  281,  282,  283,  284,  285,  286,  287,  288,
         289,  290,  291,  292,  293,  294,  295,  296,  297,  298,  299,
         300,  301,  302,  303,  304,  305,  306,  307,  308,  309,  310,
         311,  312,  313,  314,  315,  316,  317,  318,  319,  320,  321,
         322,  323,  324,  325,  326,  327,  328,  329,  330,  331,  332,
         333,  334,  335,  336,  337,  338,  339,  340,  341,  342,  343,
         344,  345,  346,  347,  348, 

## Using the model

In [18]:
def get_input_features():
    return list(zip(callset['variants/CHROM'], callset['variants/POS'], callset['variants/ALT'][:,0]))

input_features = get_input_features()
input_features

[('Pf3D7_01_v3', np.int32(489337), 'C'),
 ('Pf3D7_02_v3', np.int32(305438), 'G'),
 ('Pf3D7_02_v3', np.int32(367271), 'T'),
 ('Pf3D7_02_v3', np.int32(375427), 'A'),
 ('Pf3D7_03_v3', np.int32(383169), 'T'),
 ('Pf3D7_03_v3', np.int32(617426), 'A'),
 ('Pf3D7_03_v3', np.int32(619609), 'G'),
 ('Pf3D7_03_v3', np.int32(637880), 'A'),
 ('Pf3D7_03_v3', np.int32(637882), 'T'),
 ('Pf3D7_03_v3', np.int32(705863), 'T'),
 ('Pf3D7_03_v3', np.int32(788638), 'T'),
 ('Pf3D7_04_v3', np.int32(306270), 'A'),
 ('Pf3D7_04_v3', np.int32(329484), 'A'),
 ('Pf3D7_04_v3', np.int32(465547), 'T'),
 ('Pf3D7_08_v3', np.int32(849645), 'C'),
 ('Pf3D7_08_v3', np.int32(1063122), 'G'),
 ('Pf3D7_09_v3', np.int32(272202), 'T'),
 ('Pf3D7_09_v3', np.int32(518294), 'G'),
 ('Pf3D7_09_v3', np.int32(596674), 'C'),
 ('Pf3D7_09_v3', np.int32(664460), 'C'),
 ('Pf3D7_09_v3', np.int32(973529), 'A'),
 ('Pf3D7_09_v3', np.int32(1179855), 'C'),
 ('Pf3D7_09_v3', np.int32(1254042), 'C'),
 ('Pf3D7_09_v3', np.int32(1258030), 'A'),
 ('Pf3D7_09_

In [ ]:
variants = {}
import pysam
vcf = pysam.VariantFile('test.vcf')
for var in vcf:
    key = (var.chrom, var.pos,var.alts[0])
    if key in input_features:
        variants[key] = var.samples[0]['GT'][0]

input_data = list(variants.values())
input_data


[0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 1,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 0,
 0,
 1,
 1,
 0,
 1,
 1,
 1,
 0,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 0,
 1,
 1,
 1,
 1,
 1,
 0,
 0,
 1,
 0,
 0,
 0,
 0]

In [31]:
probs = clf.predict_proba([input_data])
tab = pd.DataFrame(zip(clf.classes_,probs[0]))
tab.columns = ['Region','Probability']
tab


,Region,Probability
0,Central Africa,0.000137
1,Eastern Africa,0.000339
2,South America,0.000035
3,South Asia,0.317619
4,Southeast Asia,0.680741
5,Western Africa,0.001129


In [38]:
class GeoPredictor:
    def __init__(self, model: HistGradientBoostingClassifier, input_features: List[tuple]):
        self.model = model
        self.input_features = input_features

    def predict(self, vcf_file: dict) -> dict:
        variants = self.get_variants_from_vcf(vcf_file)
        input_data = [variants.get(feat, np.nan) for feat in self.input_features]
        probs = self.model.predict_proba([input_data])[0]
        tab = pd.DataFrame(zip(self.model.classes_,probs))
        tab.columns = ['Region','Probability']
        return tab.sort_values(by='Probability', ascending=False)

    def get_variants_from_vcf(self, filename):
        vcf = pysam.VariantFile(filename)
        variants = {}
        for var in vcf:
            key = (var.chrom, var.pos,var.alts[0])
            if key in self.input_features:
                variants[key] = var.samples[0]['GT'][0]
        return variants

In [40]:
gp = GeoPredictor(clf, input_features)
pickle.dump(gp, open('geopredictor_model.pkl','wb'))